# Structured Outputs

*Notebook 07*

Free-form text is fragile. When agent output flows into code, a formatting variation breaks everything.

Structured outputs give you a guaranteed shape — defined once, enforced automatically.

**Topics:**
- Why free-form text breaks real pipelines

- Defining output structure with Pydantic models

- The `output_type=` parameter

- Accessing structured fields from `result.final_output`

- Validating and parsing agent output safely

- When structured outputs help most

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from agents import Agent, Runner
from typing import Optional, Annotated
from pydantic import BaseModel, Field

MODEL = "gpt-5-mini"

print("✅ Ready!")

---

## 🎯 The Problem

An agent that returns free-form text works fine for humans reading the output — but it breaks the moment another piece of code needs to use that output.

In [ ]:
# ❌ Free-form text — works for humans, breaks for code
unstructured_agent = Agent(
    name="UnstructuredAnalyst",
    instructions="Analyze the sentiment of the given text. Respond with your analysis.",
    model=MODEL
)

result = await Runner.run(unstructured_agent, input="The product arrived on time and works great. Very happy with this purchase.")

print("Raw output:")
print(result.final_output)
print()

# Try to use the output in code — this will fail unpredictably
output = result.final_output
try:
    sentiment = output["sentiment"]  # TypeError — string indices must be integers
except TypeError:
    print("❌ result.final_output is a string — there's no field to access")

### 💡 Why This Breaks

The agent might say "This is positive" or "Sentiment: positive" — all correct, none consistent. Downstream code that depends on a specific format breaks when phrasing changes.

---

## 📐 Part 1: Defining Output Structure with Pydantic

Define a Pydantic model, pass it to the agent via `output_type=`, and the SDK requests a defined schema and validates the returned shape — invalid outputs surface as errors instead of reaching your Python code in the wrong shape and breaking later.

In your own projects, define fields based on what the next step in your code needs — not everything the agent could say.

In [ ]:
class SentimentResult(BaseModel):
    sentiment: str
    confidence: Annotated[float, Field(ge=0.0, le=1.0)]
    key_phrase: str
    reasoning: str


# --------------------------------------------------------------
print("✅ SentimentResult model defined")
print(f"    Fields: {list(SentimentResult.model_fields.keys())}")

#### Run with Structured Output

In [ ]:
sentiment_instructions = (
    "Analyze the sentiment of the given text.\n"
    "Classify as positive, negative, or neutral.\n"
    "Identify the key phrase that most signals the sentiment.\n"
    "Provide a confidence score from 0.0 to 1.0."
)

sentiment_agent = Agent(
    name="SentimentAnalyst",
    instructions=sentiment_instructions,
    model=MODEL,
    output_type=SentimentResult
)

result = await Runner.run(sentiment_agent, input="The product arrived on time and works great. Very happy with this purchase.")

analysis = result.final_output

print(f"Sentiment:    {analysis.sentiment}")
print(f"Confidence:  {analysis.confidence:.0%}")
print(f"Key phrase:  '{analysis.key_phrase}'")
print(f"Reasoning:    {analysis.reasoning}")

### 💡 Why This Works

The SDK uses the Pydantic model to generate a JSON schema and instructs the model to conform to it. `result.final_output` is now an instance of `SentimentResult`, not a string. The model also uses your field names as instructions, so descriptive names like `reasoning` produce better results than short names like `rsn`.

If the model returns data that violates a `Field` constraint, the SDK raises a validation error instead of passing through invalid output.

Note: structured outputs guarantee *shape*, not truth — the model still generates the values, so validate facts separately if correctness matters.

---

## 🧩 Part 2: Optional Fields

Not every field needs to be required. Use `Optional` for fields the agent might not always have enough information to fill.

In [ ]:
class ContactInfo(BaseModel):
    name: str
    email: Optional[str] = None
    phone: Optional[str] = None
    company: Optional[str] = None
    is_complete: bool


extractor_instructions = (
    "Extract contact information from the given text.\n"
    "Only populate fields that are clearly present in the text.\n"
    "Set is_complete=True only if name and at least one contact method is present."
)

extractor_agent = Agent(
    name="ContactExtractor",
    instructions=extractor_instructions,
    model=MODEL,
    output_type=ContactInfo
)

# --------------------------------------------------------------
print("✅ ContactExtractor agent ready")

#### Extract from Multiple Texts

In [ ]:
texts = [
    "Please reach out to Sarah Chen at sarah@techcorp.com or call 555-0142.",
    "Contact Marcus for details.",
    "Email hello@startup.io to schedule a demo with our sales team."
]

print("📋 CONTACT EXTRACTION")
print("=" * 60)

for text in texts:
    result = await Runner.run(extractor_agent, input=text)

    contact = result.final_output

    display_text = f"{text[:50]}..." if len(text) > 50 else text
    print(f"\nText: \"{display_text}\"")
    print(f"  Name:      {contact.name}")
    print(f"  Email:     {contact.email or '—'}")
    print(f"  Phone:     {contact.phone or '—'}")
    print(f"  Complete: {'✅' if contact.is_complete else '⚠️  Incomplete'}")

### 💡 Why This Works

`Optional` fields make missing data explicit — downstream code can safely distinguish "absent" from "wrong" without string parsing. Use this pattern when real input is messy — resumes, emails, notes, and forms often contain some fields but not others.

## 💪 Practice Exercises

### Exercise 1: Meeting Notes Extractor

*Covers: `output_type` — extracting structured data from unstructured text*

Build an agent that extracts structured information from raw meeting notes.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Meeting Notes Extractor
# --------------------------------------------------------------
# Objective: Extract structured data from unstructured meeting notes.

meeting_notes = """
Weekly sync — Jan 15
Attendees: Priya, Tom, Leila
We decided to launch the beta on Feb 1st. Tom will handle the
infrastructure setup by Jan 25. Priya needs to finish the onboarding
flow before that. Leila is out next week so the design review moves to Jan 22.
"""

# TODO 1: Define a MeetingNotes Pydantic model with fields for:
#            - date (str)
#            - attendees (list[str])
#            - decisions (list[str])
#            - action_items (list[str])  — each as "Person: task by date"
#            - next_meeting (Optional[str])

# TODO 2: Create an agent with output_type=MeetingNotes
#            Instructions: extract all structured information from meeting notes

# TODO 3: Run the agent on meeting_notes above

# TODO 4: Print each field clearly — show that you can access
#            attendees[0], action_items[1], etc. without string parsing

# --- Write your code below this line ---

### Exercise 2: Content Classifier

*Covers: `output_type` — classification with structured reasoning*

Build an agent that classifies text content and explains its reasoning.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Content Classifier
# --------------------------------------------------------------
# Objective: Classify content and confirm downstream code can use the result.

samples = [
    "The new onboarding flow reduced user drop-off by 40% in the first week.",
    "We need to reschedule the Q3 planning meeting — Friday no longer works for the team.",
    "The API documentation is missing examples for the authentication endpoints."
]

# TODO 1: Define a ContentClassification Pydantic model with:
#            - category (str)      — e.g. "metrics", "scheduling", "documentation", "support", "other"
#            - subcategory (str)   — more specific label
#            - confidence (float)  — 0.0 to 1.0
#            - reasoning (str)     — one sentence explanation

# TODO 2: Create a classifier agent with output_type=ContentClassification

# TODO 3: Run the agent on each sample in a loop

# TODO 4: Group results by category and print a summary
#            e.g. "metrics: 1 sample, scheduling: 1 sample, documentation: 1 sample"
#            This only works cleanly because the output is structured

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**`output_type=` enforces structure:**

- Pass any Pydantic model — the SDK generates the schema and enforces it
- `result.final_output` is an instance of your model, not a string
- Access fields with dot notation: `result.final_output.sentiment`

**Structured outputs enable real pipelines:**

- Downstream code uses fields directly — no string parsing, no regex
- Aggregate operations (`sum`, `filter`, `group_by`) work immediately
- Format variations between runs no longer break anything

**Design your models carefully:**

- Use `Optional[type] = None` for fields the agent might not have data for
- Keep field names clear and descriptive — the model reads them as instructions
- Keep models minimal — add fields only when the downstream code actually needs them
- **Use structured outputs whenever the response feeds into code** — free-form text is the right choice for human-facing chat replies

---

## 📍 Next Step

08: Error Handling & Recovery — Learn what happens when tools fail and how to write agents that recover gracefully instead of crashing.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-07-structured-outputs)

---